# Tests de mini-funcionalidades de OP-07 `read_trips`

Este notebook se usa para probar helpers y bloques internos de `read_trips()` antes de hacer smoke tests o tests integrados de la función pública completa.

Objetivo:
- verificar minifuncionalidades de lectura de forma aislada;
- cubrir layout formal de trips;
- cubrir sidecar `trips.metadata.json`;
- cubrir resolución backend-aware de Parquet y Feather;
- cubrir reconstrucción de schema y schema_effective;
- cubrir lectura tabular;
- cubrir recuperación de metadata, identidades y estado de validación;
- dejar una base fácil de portar después a pytest.

Convenciones:
- los tests usan `assert`;
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`;
- las pruebas de este notebook no reemplazan smoke tests ni integration tests;
- este notebook cubre solo OP-07 `read_trips`, no OP-06 `write_trips`.

## Bloque 0. Preparación

### 0.1 Imports generales

Qué prepara: imports básicos, JSON, filesystem, pandas y PyArrow para construir artefactos mínimos de prueba.

Nota: PyArrow se usa aquí para preparar archivos Parquet/Feather que luego serán leídos por helpers de OP-07.

In [2]:
import copy
import json
import shutil
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather

### 0.2 Imports del módulo

Qué prepara: imports de clases y helpers reales usados por OP-07 `read_trips`.

Importante:
- no se importa `write_trips`;
- no se importan helpers exclusivos de escritura como staging, commit o write table;
- las funciones de escritura física que aparezcan en fixtures son solo setup de prueba, no subject under test.

In [3]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.datasets import TripDataset
from pylondrina.reports import Issue
from pylondrina.errors import ExportError

from pylondrina.io.trips import (
    ReadTripsOptions,
    _validate_read_root_and_sidecar,
    _resolve_trip_data_path_from_sidecar,
    _load_sidecar_json,
    _extract_storage_format,
    _resolve_read_schema_state,
    _read_trips_table_from_storage,
    _finalize_loaded_metadata_state,
    _build_read_trips_summary,
    _resolve_trips_artifact_root_for_read,
    _trip_data_filename_for_storage,
    _resolve_trips_artifact_paths,
    _assert_json_safe,
    _trip_schema_to_snapshot,
    _trip_schema_effective_to_snapshot,
    _build_issues_summary,
    _build_io_event,
    _append_event,
    _options_to_read_parameters,
    _compare_schema_snapshots,
    _extract_correspondence_from_metadata,
)

### 0.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para assertions y mensajes, siguiendo el estilo del notebook original.

In [5]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"


### 0.4 Configuración visual

In [6]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


### 0.5 Carpeta visible para artefactos helper-level

Qué prepara: una carpeta local, junto al notebook, para crear artefactos mínimos de lectura.

La carpeta se reinicia al ejecutar esta celda.

In [7]:
HELPER_ROOT = Path("./tmp_op07_read_trips_helper")


def reset_helper_root() -> Path:
    if HELPER_ROOT.exists():
        shutil.rmtree(HELPER_ROOT)
    HELPER_ROOT.mkdir(parents=True, exist_ok=True)
    return HELPER_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = HELPER_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_helper_root()
print("HELPER_ROOT =", root.resolve())
show_ok("Carpeta local de helper-level preparada")

HELPER_ROOT = C:\projects\pylondrina\notebooks\testing\io_trips\tmp_op07_read_trips_helper
OK - Carpeta local de helper-level preparada


## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para schema, schema_effective, dataframe, sidecar y artefactos formales mínimos.

Estas fixtures no prueban `write_trips`; solo crean archivos mínimos para poder testear helpers de OP-07.

In [8]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )


def make_trip_schema_effective(
    *,
    dtype_effective: dict | None = None,
    overrides: dict | None = None,
    domains_effective: dict | None = None,
    temporal: dict | None = None,
    fields_effective: list | None = None,
) -> TripSchemaEffective:
    return TripSchemaEffective(
        dtype_effective=dtype_effective or {},
        overrides=overrides or {},
        domains_effective=domains_effective or {},
        temporal=temporal or {},
        fields_effective=fields_effective or [],
    )


def make_trip_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "movement_id": ["m1", "m2", "m3"],
            "trip_id": ["t1", "t2", "t3"],
            "movement_seq": [0, 0, 0],
            "user_id": ["u1", "u2", "u3"],
            "origin_latitude": [-33.45, -33.46, -33.47],
            "origin_longitude": [-70.66, -70.67, -70.68],
            "destination_latitude": [-33.41, -33.42, -33.43],
            "destination_longitude": [-70.61, -70.62, -70.63],
            "mode": ["bus", "metro", "bus"],
            "purpose": ["work", "study", "work"],
            "comment": ["a", "b", "c"],
            "trip_weight": [1.0, 2.5, 1.2],
        }
    )


def make_trip_schema_minimal() -> TripSchema:
    return make_trip_schema(
        [
            make_field("movement_id", "string", required=True),
            make_field("trip_id", "string", required=True),
            make_field("movement_seq", "int", required=True),
            make_field("user_id", "string", required=True),
            make_field("origin_latitude", "float", required=True),
            make_field("origin_longitude", "float", required=True),
            make_field("destination_latitude", "float", required=True),
            make_field("destination_longitude", "float", required=True),
            make_field(
                "mode",
                "categorical",
                required=False,
                domain=DomainSpec(values=["bus", "metro", "walk", "car"], extendable=True),
            ),
            make_field(
                "purpose",
                "categorical",
                required=False,
                domain=DomainSpec(values=["work", "study", "health"], extendable=True),
            ),
            make_field("comment", "string", required=False),
            make_field("trip_weight", "float", required=False),
        ]
    )


def make_trip_schema_effective_minimal() -> TripSchemaEffective:
    return make_trip_schema_effective(
        dtype_effective={
            "mode": "categorical",
            "purpose": "categorical",
            "trip_weight": "float",
        },
        domains_effective={
            "mode": {"values": ["bus", "metro", "walk", "car"]},
            "purpose": {"values": ["work", "study", "health"]},
        },
        temporal={"tier": "tier_3"},
        fields_effective=[
            "movement_id",
            "trip_id",
            "movement_seq",
            "user_id",
            "origin_latitude",
            "origin_longitude",
            "destination_latitude",
            "destination_longitude",
            "mode",
            "purpose",
            "comment",
            "trip_weight",
        ],
    )


def make_storage_options_snapshot(
    *,
    storage_format: str = "parquet",
    parquet_compression: str | None = "snappy",
    feather_compression: str | None = "lz4",
) -> dict:
    if storage_format == "parquet":
        return {"compression": parquet_compression}
    if storage_format == "feather":
        return {"compression": feather_compression, "version": 2}
    return {"compression": None}


def make_sidecar_payload(
    *,
    schema: TripSchema | None = None,
    schema_effective: TripSchemaEffective | None = None,
    metadata: dict | None = None,
    provenance: dict | None = None,
    dataset_id: str = "dset_sidecar_001",
    artifact_id: str | None = "art_sidecar_001",
    storage_format: str = "parquet",
    parquet_compression: str | None = "snappy",
    feather_compression: str | None = "lz4",
    include_schema_effective: bool = True,
) -> dict:
    schema = schema or make_trip_schema_minimal()
    schema_effective = schema_effective or make_trip_schema_effective_minimal()

    metadata = copy.deepcopy(metadata) if metadata is not None else {
        "dataset_id": dataset_id,
        "artifact_id": artifact_id,
        "is_validated": True,
        "events": [],
        "mappings": {
            "field_correspondence": {"movement_id": "movement_id_src"},
            "value_correspondence": {"mode": {"micro": "bus"}},
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
    }

    provenance = copy.deepcopy(provenance) if provenance is not None else {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    payload = {
        "dataset_type": "trips",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": storage_format,
            "options": make_storage_options_snapshot(
                storage_format=storage_format,
                parquet_compression=parquet_compression,
                feather_compression=feather_compression,
            ),
        },
        "dataset_id": dataset_id,
        "artifact_id": artifact_id,
        "files": {
            "data": _trip_data_filename_for_storage(storage_format),
            "metadata": "trips.metadata.json",
        },
        "schema": _trip_schema_to_snapshot(schema),
        "provenance": provenance,
        "metadata": metadata,
    }

    if include_schema_effective:
        payload["schema_effective"] = _trip_schema_effective_to_snapshot(schema_effective)

    return payload


def write_table_for_read_fixture(
    df: pd.DataFrame,
    data_path: Path,
    *,
    storage_format: str,
    parquet_compression: str | None = "snappy",
    feather_compression: str | None = "lz4",
) -> None:
    """
    Escribe el archivo tabular mínimo para fixtures de lectura.
    Esto es setup del test, no subject under test.
    """
    require_pyarrow()

    if storage_format == "parquet":
        df.to_parquet(
            data_path,
            index=False,
            engine="pyarrow",
            compression=None if parquet_compression == "none" else parquet_compression,
        )
        return

    if storage_format == "feather":
        table = pa.Table.from_pandas(df, preserve_index=False)
        feather.write_feather(
            table,
            data_path,
            compression=feather_compression,
            version=2,
        )
        return

    raise ValueError(f"storage_format no soportado en fixture: {storage_format!r}")


def materialize_minimal_formal_artifact(
    root_dir: Path,
    *,
    df: pd.DataFrame | None = None,
    schema: TripSchema | None = None,
    schema_effective: TripSchemaEffective | None = None,
    metadata: dict | None = None,
    provenance: dict | None = None,
    dataset_id: str = "dset_artifact_001",
    artifact_id: str | None = "art_artifact_001",
    storage_format: str = "parquet",
    parquet_compression: str | None = "snappy",
    feather_compression: str | None = "lz4",
    include_schema_effective: bool = True,
) -> tuple:
    """
    Crea en disco un artefacto formal mínimo para tests de lectura.
    Esto es setup de pruebas, no el subject under test.
    """
    require_pyarrow()

    root_dir.mkdir(parents=True, exist_ok=True)

    df = df.copy() if df is not None else make_trip_df()
    schema = schema or make_trip_schema_minimal()
    schema_effective = schema_effective or make_trip_schema_effective_minimal()

    data_filename = _trip_data_filename_for_storage(storage_format)
    data_path = root_dir / data_filename
    sidecar_path = root_dir / "trips.metadata.json"

    write_table_for_read_fixture(
        df,
        data_path,
        storage_format=storage_format,
        parquet_compression=parquet_compression,
        feather_compression=feather_compression,
    )

    payload = make_sidecar_payload(
        schema=schema,
        schema_effective=schema_effective,
        metadata=metadata,
        provenance=provenance,
        dataset_id=dataset_id,
        artifact_id=artifact_id,
        storage_format=storage_format,
        parquet_compression=parquet_compression,
        feather_compression=feather_compression,
        include_schema_effective=include_schema_effective,
    )

    sidecar_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return _resolve_trips_artifact_paths(root_dir), payload

## Bloque 2. Helpers de path, layout y sidecar

Qué prueba: resolución de rutas, fallback `.golondrina`, validación del root formal y carga del sidecar obligatorio.

### Test 2.1 - `_resolve_trips_artifact_paths`

Qué prueba: resolución estable del root, sidecar oficial y sidecar legacy.

In [9]:
root = Path("tmp_demo_artifact")
paths = _resolve_trips_artifact_paths(root)

assert paths.root_dir == root
assert paths.sidecar_path == root / "trips.metadata.json"
assert paths.legacy_sidecar_path == root / "metadata.json"

assert not hasattr(paths, "data_path"), (
    "En la implementación vigente, TripsArtifactPaths no debe asumir data_path fijo; "
    "el archivo tabular se resuelve desde sidecar['files']['data']."
)

show_ok("Test 2.1 - _resolve_trips_artifact_paths")

OK - Test 2.1 - _resolve_trips_artifact_paths


### Test 2.2 - `_resolve_trips_artifact_root_for_read`

Qué prueba:
- si el path exacto existe, se usa tal cual;
- si no existe y existe `path.golondrina`, se usa el bundle canónico;
- si no existe ningún candidato, se devuelve el path original para que el helper de layout emita el error.

In [10]:
case_dir = make_case_dir("test_02_02_resolve_root_for_read")

exact_root = case_dir / "exact_artifact"
exact_root.mkdir()

assert _resolve_trips_artifact_root_for_read(exact_root) == exact_root

base_path = case_dir / "canonical_artifact"
canonical_path = case_dir / "canonical_artifact.golondrina"
canonical_path.mkdir()

assert _resolve_trips_artifact_root_for_read(base_path) == canonical_path
assert _resolve_trips_artifact_root_for_read(canonical_path) == canonical_path

missing_path = case_dir / "missing_artifact"
assert _resolve_trips_artifact_root_for_read(missing_path) == missing_path

show_ok("Test 2.2 - _resolve_trips_artifact_root_for_read")

OK - Test 2.2 - _resolve_trips_artifact_root_for_read


### Test 2.3 - `_validate_read_root_and_sidecar` happy path

Qué prueba: layout formal mínimo correcto con sidecar oficial.

In [11]:
case_dir = make_case_dir("test_02_03_validate_root_happy")
paths, payload = materialize_minimal_formal_artifact(case_dir / "artifact")

issues = []

_validate_read_root_and_sidecar(
    paths.root_dir,
    paths,
    issues=issues,
)

assert issues == []

show_ok("Test 2.3 - _validate_read_root_and_sidecar happy path")

OK - Test 2.3 - _validate_read_root_and_sidecar happy path


### Test 2.4 - `_validate_read_root_and_sidecar` fatal por root inválido

Qué prueba: si el root no existe o no es directorio, OP-07 aborta antes de intentar leer sidecar o tabla.

In [12]:
case_dir = make_case_dir("test_02_04_validate_root_invalid")
missing_root = case_dir / "missing_artifact"
paths = _resolve_trips_artifact_paths(missing_root)

issues = []

try:
    _validate_read_root_and_sidecar(
        missing_root,
        paths,
        issues=issues,
    )
    raise AssertionError("Debió fallar por root inválido")
except ExportError:
    assert_issue_present(issues, "READ.PATH.INVALID_ROOT")

show_ok("Test 2.4 - _validate_read_root_and_sidecar root inválido")

OK - Test 2.4 - _validate_read_root_and_sidecar root inválido


### Test 2.5 - `_validate_read_root_and_sidecar` fatal por sidecar faltante

Qué prueba: el sidecar `trips.metadata.json` es obligatorio para lectura formal.

In [13]:
case_dir = make_case_dir("test_02_05_missing_sidecar")
root = case_dir / "artifact"
root.mkdir(parents=True)

(root / "trips.parquet").write_text("placeholder", encoding="utf-8")

paths = _resolve_trips_artifact_paths(root)
issues = []

try:
    _validate_read_root_and_sidecar(
        root,
        paths,
        issues=issues,
    )
    raise AssertionError("Debió fallar por sidecar faltante")
except ExportError:
    assert_issue_present(issues, "READ.LAYOUT.MISSING_SIDECAR")

show_ok("Test 2.5 - _validate_read_root_and_sidecar missing sidecar")

OK - Test 2.5 - _validate_read_root_and_sidecar missing sidecar


### Test 2.6 - `_validate_read_root_and_sidecar` fatal por sidecar legacy

Qué prueba: rechazo explícito de `metadata.json` cuando falta el sidecar formal `trips.metadata.json`.

In [14]:
case_dir = make_case_dir("test_02_06_legacy_sidecar")
root = case_dir / "artifact"
root.mkdir(parents=True)

(root / "trips.parquet").write_text("placeholder", encoding="utf-8")
(root / "metadata.json").write_text("{}", encoding="utf-8")

paths = _resolve_trips_artifact_paths(root)
issues = []

try:
    _validate_read_root_and_sidecar(
        root,
        paths,
        issues=issues,
    )
    raise AssertionError("Debió fallar por sidecar legacy")
except ExportError:
    assert_issue_present(issues, "READ.LAYOUT.LEGACY_SIDECAR_DETECTED")

show_ok("Test 2.6 - _validate_read_root_and_sidecar legacy sidecar")

OK - Test 2.6 - _validate_read_root_and_sidecar legacy sidecar


### Test 2.7 - `_load_sidecar_json` happy path

Qué prueba: carga de `trips.metadata.json` como payload dict.

In [15]:
case_dir = make_case_dir("test_02_07_load_sidecar_happy")
paths, payload = materialize_minimal_formal_artifact(case_dir / "artifact")

issues = []

loaded = _load_sidecar_json(
    paths.sidecar_path,
    issues=issues,
    destination_path=paths.root_dir,
)

assert loaded["dataset_type"] == "trips"
assert loaded["format"] == "golondrina"
assert loaded["layout_version"] == "1.1"
assert loaded["storage"]["format"] == "parquet"
assert issues == []

show_ok("Test 2.7 - _load_sidecar_json happy path")

OK - Test 2.7 - _load_sidecar_json happy path


### Test 2.8 - `_load_sidecar_json` fatal por JSON inválido

Qué prueba: si el sidecar existe pero no se puede parsear como JSON, la lectura formal aborta.

In [16]:
case_dir = make_case_dir("test_02_08_sidecar_invalid_json")
root = case_dir / "artifact"
root.mkdir(parents=True)

sidecar_path = root / "trips.metadata.json"
sidecar_path.write_text("{ invalid json", encoding="utf-8")

issues = []

try:
    _load_sidecar_json(
        sidecar_path,
        issues=issues,
        destination_path=root,
    )
    raise AssertionError("Debió fallar por JSON inválido")
except ExportError:
    assert_issue_present(issues, "READ.JSON.LOAD_FAILED")

show_ok("Test 2.8 - _load_sidecar_json invalid JSON")

OK - Test 2.8 - _load_sidecar_json invalid JSON


### Test 2.9 - `_load_sidecar_json` fatal por estructura top-level incompleta

Qué prueba: el sidecar debe tener las claves top-level mínimas del contrato formal.

In [17]:
case_dir = make_case_dir("test_02_09_sidecar_invalid_top_level")
root = case_dir / "artifact"
root.mkdir(parents=True)

sidecar_path = root / "trips.metadata.json"
sidecar_path.write_text(
    json.dumps({"dataset_type": "trips"}, ensure_ascii=False),
    encoding="utf-8",
)

issues = []

try:
    _load_sidecar_json(
        sidecar_path,
        issues=issues,
        destination_path=root,
    )
    raise AssertionError("Debió fallar por top-level incompleto")
except ExportError:
    assert_issue_present(issues, "READ.SIDECAR.INVALID_TOP_LEVEL")

show_ok("Test 2.9 - _load_sidecar_json top-level inválido")

OK - Test 2.9 - _load_sidecar_json top-level inválido


## Bloque 3. Helpers de backend y archivo tabular

Qué prueba: extracción de `storage.format`, coherencia `storage.format` ↔ `files.data` y lectura física desde Parquet/Feather.

### Test 3.1 - `_extract_storage_format`

Qué prueba: extracción válida de backends soportados y rechazo de backend no soportado.

In [18]:
payload_parquet = make_sidecar_payload(storage_format="parquet")
payload_feather = make_sidecar_payload(storage_format="feather")

issues = []
assert _extract_storage_format(payload_parquet, strict=False, issues=issues) == "parquet"
assert _extract_storage_format(payload_feather, strict=False, issues=issues) == "feather"
assert issues == []

payload_bad = make_sidecar_payload(storage_format="parquet")
payload_bad["storage"]["format"] = "csv"

issues_bad = []
try:
    _extract_storage_format(payload_bad, strict=False, issues=issues_bad)
    raise AssertionError("Debió fallar por storage_format no soportado")
except ExportError:
    assert_issue_present(issues_bad, "READ.STORAGE.UNSUPPORTED_FORMAT")

show_ok("Test 3.1 - _extract_storage_format")

OK - Test 3.1 - _extract_storage_format


### Test 3.2 - `_resolve_trip_data_path_from_sidecar` happy path Parquet

Qué prueba: resolución del archivo `trips.parquet` desde `files.data` y `storage.format`.

In [19]:
case_dir = make_case_dir("test_03_02_resolve_data_path_parquet")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="parquet",
)

issues = []

data_path = _resolve_trip_data_path_from_sidecar(
    paths.root_dir,
    payload,
    storage_format="parquet",
    strict=False,
    issues=issues,
)

assert data_path == paths.root_dir / "trips.parquet"
assert data_path.exists()
assert issues == []

show_ok("Test 3.2 - _resolve_trip_data_path_from_sidecar parquet")

OK - Test 3.2 - _resolve_trip_data_path_from_sidecar parquet


### Test 3.3 - `_resolve_trip_data_path_from_sidecar` happy path Feather

Qué prueba: resolución del archivo `trips.feather` desde `files.data` y `storage.format`.

In [20]:
case_dir = make_case_dir("test_03_03_resolve_data_path_feather")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="feather",
)

issues = []

data_path = _resolve_trip_data_path_from_sidecar(
    paths.root_dir,
    payload,
    storage_format="feather",
    strict=False,
    issues=issues,
)

assert data_path == paths.root_dir / "trips.feather"
assert data_path.exists()
assert issues == []

show_ok("Test 3.3 - _resolve_trip_data_path_from_sidecar feather")

OK - Test 3.3 - _resolve_trip_data_path_from_sidecar feather


### Test 3.4 - `_resolve_trip_data_path_from_sidecar` usa nombre esperado si falta `files.data`

Qué prueba: si el sidecar no declara `files.data`, el helper usa el nombre esperado para el backend.

In [21]:
case_dir = make_case_dir("test_03_04_resolve_data_path_default_filename")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="parquet",
)

payload["files"].pop("data")

issues = []

data_path = _resolve_trip_data_path_from_sidecar(
    paths.root_dir,
    payload,
    storage_format="parquet",
    strict=False,
    issues=issues,
)

assert data_path == paths.root_dir / "trips.parquet"
assert data_path.exists()
assert issues == []

show_ok("Test 3.4 - _resolve_trip_data_path_from_sidecar default filename")

OK - Test 3.4 - _resolve_trip_data_path_from_sidecar default filename


### Test 3.5 - `_resolve_trip_data_path_from_sidecar` fatal por mismatch archivo/backend

Qué prueba: si `storage.format="feather"` pero `files.data="trips.parquet"`, el artefacto se considera incoherente.

In [22]:
case_dir = make_case_dir("test_03_05_data_file_mismatch")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="feather",
)

payload["files"]["data"] = "trips.parquet"

issues = []

try:
    _resolve_trip_data_path_from_sidecar(
        paths.root_dir,
        payload,
        storage_format="feather",
        strict=False,
        issues=issues,
    )
    raise AssertionError("Debió fallar por mismatch entre storage.format y files.data")
except ExportError:
    assert_issue_present(issues, "READ.LAYOUT.DATA_FILE_MISMATCH")

show_ok("Test 3.5 - _resolve_trip_data_path_from_sidecar mismatch")

OK - Test 3.5 - _resolve_trip_data_path_from_sidecar mismatch


### Test 3.6 - `_resolve_trip_data_path_from_sidecar` fatal por archivo tabular faltante

Qué prueba: si el sidecar declara un archivo esperado pero este no existe en el bundle, la lectura aborta.

In [23]:
case_dir = make_case_dir("test_03_06_missing_data_file")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="parquet",
)

(paths.root_dir / "trips.parquet").unlink()

issues = []

try:
    _resolve_trip_data_path_from_sidecar(
        paths.root_dir,
        payload,
        storage_format="parquet",
        strict=False,
        issues=issues,
    )
    raise AssertionError("Debió fallar por archivo de datos faltante")
except ExportError:
    assert_issue_present(issues, "READ.LAYOUT.MISSING_DATA_FILE")

show_ok("Test 3.6 - _resolve_trip_data_path_from_sidecar missing data file")

OK - Test 3.6 - _resolve_trip_data_path_from_sidecar missing data file


### Test 3.7 - `_read_trips_table_from_storage` con Parquet

Qué prueba: lectura física de `trips.parquet` desde el backend declarado.

In [24]:
case_dir = make_case_dir("test_03_07_read_table_parquet")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="parquet",
)

issues = []

df_loaded = _read_trips_table_from_storage(
    paths.root_dir / "trips.parquet",
    storage_format="parquet",
    issues=issues,
    destination_path=paths.root_dir,
)

assert len(df_loaded) == 3
assert list(df_loaded.columns) == list(make_trip_df().columns)
assert issues == []

show_ok("Test 3.7 - _read_trips_table_from_storage parquet")

OK - Test 3.7 - _read_trips_table_from_storage parquet


### Test 3.8 - `_read_trips_table_from_storage` con Feather

Qué prueba: lectura física de `trips.feather` desde el backend declarado.

In [25]:
case_dir = make_case_dir("test_03_08_read_table_feather")
paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    storage_format="feather",
)

issues = []

df_loaded = _read_trips_table_from_storage(
    paths.root_dir / "trips.feather",
    storage_format="feather",
    issues=issues,
    destination_path=paths.root_dir,
)

assert len(df_loaded) == 3
assert list(df_loaded.columns) == list(make_trip_df().columns)
assert issues == []

show_ok("Test 3.8 - _read_trips_table_from_storage feather")

OK - Test 3.8 - _read_trips_table_from_storage feather


### Test 3.9 - `_read_trips_table_from_storage` emite info para dataframe vacío

Qué prueba: un artefacto formal puede reconstruir una tabla vacía, pero debe dejar evidencia informativa.

In [26]:
case_dir = make_case_dir("test_03_09_read_empty_dataframe")
empty_df = make_trip_df().iloc[0:0].copy()

paths, payload = materialize_minimal_formal_artifact(
    case_dir / "artifact",
    df=empty_df,
    storage_format="parquet",
)

issues = []

df_loaded = _read_trips_table_from_storage(
    paths.root_dir / "trips.parquet",
    storage_format="parquet",
    issues=issues,
    destination_path=paths.root_dir,
)

assert len(df_loaded) == 0
assert list(df_loaded.columns) == list(empty_df.columns)
assert_issue_present(issues, "READ.CORE.EMPTY_DATAFRAME")

show_ok("Test 3.9 - _read_trips_table_from_storage empty dataframe")

OK - Test 3.9 - _read_trips_table_from_storage empty dataframe


### Test 3.10 - `_read_trips_table_from_storage` fatal por Parquet corrupto

Qué prueba: si `storage_format="parquet"` pero el archivo no es legible como Parquet, se emite el issue correcto.

In [27]:
case_dir = make_case_dir("test_03_10_read_corrupt_parquet")
root = case_dir / "artifact"
root.mkdir(parents=True)

bad_path = root / "trips.parquet"
bad_path.write_text("not a parquet file", encoding="utf-8")

issues = []

try:
    _read_trips_table_from_storage(
        bad_path,
        storage_format="parquet",
        issues=issues,
        destination_path=root,
    )
    raise AssertionError("Debió fallar por Parquet corrupto")
except ExportError:
    assert_issue_present(issues, "READ.PARQUET.LOAD_FAILED")

show_ok("Test 3.10 - _read_trips_table_from_storage corrupt parquet")

OK - Test 3.10 - _read_trips_table_from_storage corrupt parquet


### Test 3.11 - `_read_trips_table_from_storage` fatal por Feather corrupto

Qué prueba: si `storage_format="feather"` pero el archivo no es legible como Feather, se emite el issue correcto.

In [28]:
case_dir = make_case_dir("test_03_11_read_corrupt_feather")
root = case_dir / "artifact"
root.mkdir(parents=True)

bad_path = root / "trips.feather"
bad_path.write_text("not a feather file", encoding="utf-8")

issues = []

try:
    _read_trips_table_from_storage(
        bad_path,
        storage_format="feather",
        issues=issues,
        destination_path=root,
    )
    raise AssertionError("Debió fallar por Feather corrupto")
except ExportError:
    assert_issue_present(issues, "READ.FEATHER.LOAD_FAILED")

show_ok("Test 3.11 - _read_trips_table_from_storage corrupt feather")

OK - Test 3.11 - _read_trips_table_from_storage corrupt feather


## Bloque 4. Helpers de reconstrucción de schema

Qué prueba: precedencia de `options.schema`, reconstrucción desde sidecar, mismatch, schema no disponible y recuperación de `schema_effective`.

### Test 4.1 - `_resolve_read_schema_state` usa schema desde metadata

Qué prueba: si no se entrega `options.schema`, se reconstruye el `TripSchema` desde el snapshot persistido.

In [29]:
schema_metadata = make_trip_schema_minimal()
payload = make_sidecar_payload(schema=schema_metadata)

state = _resolve_read_schema_state(
    payload,
    ReadTripsOptions(schema=None, strict=False, keep_metadata=True),
)

assert isinstance(state.schema, TripSchema)
assert state.schema.version == schema_metadata.version
assert state.schema_source == "metadata"
assert state.schema_mismatch is False
assert isinstance(state.schema_effective, TripSchemaEffective)
assert state.issues == []

show_ok("Test 4.1 - _resolve_read_schema_state desde metadata")

OK - Test 4.1 - _resolve_read_schema_state desde metadata


### Test 4.2 - `_resolve_read_schema_state` con precedencia de `options.schema`

Qué prueba: si el usuario entrega schema explícito, este tiene precedencia sobre el snapshot persistido. Si hay diferencias, queda issue de mismatch.

In [30]:
schema_metadata = make_trip_schema_minimal()
schema_options = make_trip_schema(
    [
        make_field("movement_id", "string", required=True),
        make_field("trip_id", "string", required=True),
    ],
    version="9.9",
)

payload = make_sidecar_payload(schema=schema_metadata)

state = _resolve_read_schema_state(
    payload,
    ReadTripsOptions(schema=schema_options, strict=False, keep_metadata=True),
)

assert state.schema.version == "9.9"
assert state.schema_source == "options"
assert state.schema_mismatch is True
assert_issue_present(state.issues, "READ.SCHEMA.MISMATCH")

show_ok("Test 4.2 - _resolve_read_schema_state precedence + mismatch")

OK - Test 4.2 - _resolve_read_schema_state precedence + mismatch


### Test 4.3 - `_resolve_read_schema_state` fatal por mismatch en strict=True

Qué prueba: el mismo mismatch de schema pasa de warning recuperable a error fatal cuando `strict=True`.

In [31]:
schema_metadata = make_trip_schema_minimal()
schema_options = make_trip_schema(
    [
        make_field("movement_id", "string", required=True),
        make_field("trip_id", "string", required=True),
    ],
    version="9.9",
)

payload = make_sidecar_payload(schema=schema_metadata)

try:
    _resolve_read_schema_state(
        payload,
        ReadTripsOptions(schema=schema_options, strict=True, keep_metadata=True),
    )
    raise AssertionError("Debió fallar por schema mismatch con strict=True")
except ExportError as e:
    assert e.code == "READ.SCHEMA.MISMATCH"

show_ok("Test 4.3 - _resolve_read_schema_state strict schema mismatch")

OK - Test 4.3 - _resolve_read_schema_state strict schema mismatch


### Test 4.4 - `_resolve_read_schema_state` ignora metadata schema inválido si hay `options.schema`

Qué prueba: si el snapshot de schema persistido no es interpretable, pero el usuario entrega un schema usable, la lectura puede continuar con warning.

In [32]:
schema_options = make_trip_schema_minimal()

payload = make_sidecar_payload()
payload["schema"] = {"version": "bad", "fields": "not-a-mapping"}

state = _resolve_read_schema_state(
    payload,
    ReadTripsOptions(schema=schema_options, strict=False, keep_metadata=True),
)

assert state.schema is schema_options
assert state.schema_source == "options"
assert state.schema_mismatch is False
assert_issue_present(state.issues, "READ.SCHEMA.METADATA_INVALID_IGNORED")

show_ok("Test 4.4 - _resolve_read_schema_state metadata inválida ignorada")

OK - Test 4.4 - _resolve_read_schema_state metadata inválida ignorada


### Test 4.5 - `_resolve_read_schema_state` fatal sin schema recuperable

Qué prueba: si no hay `options.schema` ni snapshot de schema usable, la lectura no puede reconstruir el contrato.

In [33]:
payload = make_sidecar_payload()
payload.pop("schema")

try:
    _resolve_read_schema_state(
        payload,
        ReadTripsOptions(schema=None, strict=False, keep_metadata=True),
    )
    raise AssertionError("Debió fallar por schema no recuperable")
except ExportError as e:
    assert e.code == "READ.SCHEMA.UNAVAILABLE"

show_ok("Test 4.5 - _resolve_read_schema_state fatal unavailable")

OK - Test 4.5 - _resolve_read_schema_state fatal unavailable


### Test 4.6 - `_resolve_read_schema_state` con `schema_effective` faltante y recovery

Qué prueba: con `strict=False`, si falta `schema_effective`, se reconstruye un `TripSchemaEffective` default y se emite warning.

In [34]:
payload = make_sidecar_payload()
payload.pop("schema_effective")

state = _resolve_read_schema_state(
    payload,
    ReadTripsOptions(schema=None, strict=False, keep_metadata=True),
)

assert isinstance(state.schema_effective, TripSchemaEffective)
assert state.schema_effective.to_dict() == TripSchemaEffective().to_dict()
assert_issue_present(state.issues, "READ.SCHEMA_EFFECTIVE.DEFAULTED")

show_ok("Test 4.6 - _resolve_read_schema_state default schema_effective")

OK - Test 4.6 - _resolve_read_schema_state default schema_effective


### Test 4.7 - `_resolve_read_schema_state` fatal por `schema_effective` faltante con strict=True

Qué prueba: con `strict=True`, el `schema_effective` faltante o inválido no se degrada a default.

In [35]:
payload = make_sidecar_payload()
payload.pop("schema_effective")

try:
    _resolve_read_schema_state(
        payload,
        ReadTripsOptions(schema=None, strict=True, keep_metadata=True),
    )
    raise AssertionError("Debió fallar por schema_effective faltante con strict=True")
except ExportError as e:
    assert e.code == "READ.SCHEMA_EFFECTIVE.DEFAULTED"

show_ok("Test 4.7 - _resolve_read_schema_state strict schema_effective faltante")

OK - Test 4.7 - _resolve_read_schema_state strict schema_effective faltante


### Test 4.8 - `_compare_schema_snapshots`

Qué prueba: comparación mínima entre schema de options y schema persistido para detectar mismatch observable.

In [36]:
schema_a = make_trip_schema_minimal()

schema_b = make_trip_schema(
    [
        make_field("movement_id", "string", required=True),
        make_field("trip_id", "string", required=True),
    ],
    version="2.0",
)

diff = _compare_schema_snapshots(schema_b, schema_a)

assert diff["schema_mismatch"] is True
assert "movement_seq" in diff["required_diff"]
assert diff["fields_diff_total"] > 0
assert isinstance(diff["fields_diff_sample"], list)

same = _compare_schema_snapshots(schema_a, schema_a)
assert same["schema_mismatch"] is False
assert same["required_diff"] == []
assert same["fields_diff_total"] == 0

show_ok("Test 4.8 - _compare_schema_snapshots")

OK - Test 4.8 - _compare_schema_snapshots


## Bloque 5. Helpers de metadata, identidad y correspondencias

Qué prueba: recuperación de identidad lógica, tratamiento de `artifact_id`, forcing de `is_validated=False` y extracción de mappings.

### Test 5.1 - `_finalize_loaded_metadata_state` normal

Qué prueba: preserva identidad cargada y fuerza `metadata["is_validated"] = False`.

In [37]:
metadata = {
    "dataset_id": "dset_ok",
    "artifact_id": "art_ok",
    "is_validated": True,
    "events": [],
}

sidecar_payload = {
    "dataset_id": "dset_ok",
    "artifact_id": "art_ok",
}

state = _finalize_loaded_metadata_state(
    metadata,
    sidecar_payload=sidecar_payload,
    strict=False,
    destination_path=Path("/tmp/fake_artifact"),
)

assert state.dataset_id == "dset_ok"
assert state.dataset_id_status == "loaded"
assert state.artifact_id == "art_ok"
assert state.artifact_id_status == "loaded"
assert state.metadata["dataset_id"] == "dset_ok"
assert state.metadata["artifact_id"] == "art_ok"
assert state.metadata["is_validated"] is False

assert_issue_present(state.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")
assert_counts_by_level(state.issues, info=1)

show_ok("Test 5.1 - _finalize_loaded_metadata_state normal")

OK - Test 5.1 - _finalize_loaded_metadata_state normal


### Test 5.2 - `_finalize_loaded_metadata_state` prioriza IDs top-level del sidecar

Qué prueba: si el sidecar top-level trae `dataset_id` y `artifact_id`, esos valores mandan sobre los valores embebidos en metadata.

In [38]:
metadata = {
    "dataset_id": "dset_from_metadata",
    "artifact_id": "art_from_metadata",
    "is_validated": True,
    "events": [],
}

sidecar_payload = {
    "dataset_id": "dset_top_level",
    "artifact_id": "art_top_level",
}

state = _finalize_loaded_metadata_state(
    metadata,
    sidecar_payload=sidecar_payload,
    strict=False,
    destination_path=Path("/tmp/fake_artifact"),
)

assert state.dataset_id == "dset_top_level"
assert state.artifact_id == "art_top_level"
assert state.metadata["dataset_id"] == "dset_top_level"
assert state.metadata["artifact_id"] == "art_top_level"
assert state.metadata["is_validated"] is False

assert_issue_present(state.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

show_ok("Test 5.2 - _finalize_loaded_metadata_state top-level precedence")

OK - Test 5.2 - _finalize_loaded_metadata_state top-level precedence


### Test 5.3 - `_finalize_loaded_metadata_state` recovery con strict=False

Qué prueba: si faltan o son inválidos `dataset_id` y `artifact_id`, la lectura puede recuperar estado con `strict=False`.

In [39]:
metadata = {
    "is_validated": True,
    "events": [],
}

sidecar_payload = {
    "dataset_id": "",
    "artifact_id": None,
}

state = _finalize_loaded_metadata_state(
    metadata,
    sidecar_payload=sidecar_payload,
    strict=False,
    destination_path=Path("/tmp/fake_artifact"),
)

assert state.dataset_id_status == "regenerated"
assert isinstance(state.dataset_id, str) and state.dataset_id
assert state.artifact_id is None
assert state.artifact_id_status == "missing_or_invalid"
assert state.metadata["dataset_id"] == state.dataset_id
assert state.metadata["artifact_id"] is None
assert state.metadata["is_validated"] is False

assert_issue_present(state.issues, "READ.METADATA.DATASET_ID_REGENERATED")
assert_issue_present(state.issues, "READ.METADATA.ARTIFACT_ID_SET_NONE")
assert_issue_present(state.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

show_ok("Test 5.3 - _finalize_loaded_metadata_state recovery")

OK - Test 5.3 - _finalize_loaded_metadata_state recovery


### Test 5.4 - `_finalize_loaded_metadata_state` fatal por `dataset_id` inválido con strict=True

Qué prueba: en modo estricto, no se regenera `dataset_id`; se aborta.

In [40]:
metadata = {
    "is_validated": True,
    "events": [],
}

sidecar_payload = {
    "dataset_id": "",
    "artifact_id": "art_ok",
}

try:
    _finalize_loaded_metadata_state(
        metadata,
        sidecar_payload=sidecar_payload,
        strict=True,
        destination_path=Path("/tmp/fake_artifact"),
    )
    raise AssertionError("Debió fallar por dataset_id inválido con strict=True")
except ExportError as e:
    assert e.code == "READ.SIDECAR.INVALID_TOP_LEVEL"

show_ok("Test 5.4 - _finalize_loaded_metadata_state strict dataset_id inválido")

OK - Test 5.4 - _finalize_loaded_metadata_state strict dataset_id inválido


### Test 5.5 - `_finalize_loaded_metadata_state` fatal por `artifact_id` inválido con strict=True

Qué prueba: en modo estricto, no se acepta `artifact_id` faltante o inválido.

In [41]:
metadata = {
    "dataset_id": "dset_ok",
    "is_validated": True,
    "events": [],
}

sidecar_payload = {
    "dataset_id": "dset_ok",
    "artifact_id": None,
}

try:
    _finalize_loaded_metadata_state(
        metadata,
        sidecar_payload=sidecar_payload,
        strict=True,
        destination_path=Path("/tmp/fake_artifact"),
    )
    raise AssertionError("Debió fallar por artifact_id inválido con strict=True")
except ExportError as e:
    assert e.code == "READ.SIDECAR.INVALID_TOP_LEVEL"

show_ok("Test 5.5 - _finalize_loaded_metadata_state strict artifact_id inválido")

OK - Test 5.5 - _finalize_loaded_metadata_state strict artifact_id inválido


### Test 5.6 - `_extract_correspondence_from_metadata`

Qué prueba: extracción segura de `field_correspondence` y `value_correspondence` desde `metadata["mappings"]`.

In [42]:
metadata = {
    "mappings": {
        "field_correspondence": {
            "movement_id": "id_original",
            "mode": "modo_original",
        },
        "value_correspondence": {
            "mode": {
                "micro": "bus",
                "metrotren": "train",
            }
        },
    }
}

field_corr, value_corr = _extract_correspondence_from_metadata(metadata)

assert field_corr == {
    "movement_id": "id_original",
    "mode": "modo_original",
}
assert value_corr == {
    "mode": {
        "micro": "bus",
        "metrotren": "train",
    }
}

field_corr_empty, value_corr_empty = _extract_correspondence_from_metadata({})
assert field_corr_empty == {}
assert value_corr_empty == {}

field_corr_bad, value_corr_bad = _extract_correspondence_from_metadata({"mappings": "bad"})
assert field_corr_bad == {}
assert value_corr_bad == {}

show_ok("Test 5.6 - _extract_correspondence_from_metadata")

OK - Test 5.6 - _extract_correspondence_from_metadata


## Bloque 6. Helpers de reporte, parámetros y evento de lectura

Qué prueba: serialización de parámetros, summary estable, issues_summary y evento `read_trips`.

### Test 6.1 - `_options_to_read_parameters`

Qué prueba: serialización estable de `ReadTripsOptions` para report y evento.

In [44]:
params_default = _options_to_read_parameters(
    path=Path("artifact.golondrina"),
    options=ReadTripsOptions(),
)

assert params_default["path"] == str(Path("artifact.golondrina").expanduser())
assert params_default["strict"] is False
assert params_default["keep_metadata"] is True
assert params_default["schema"] is None

schema = make_trip_schema_minimal()
params_schema = _options_to_read_parameters(
    path=Path("artifact.golondrina"),
    options=ReadTripsOptions(schema=schema, strict=True, keep_metadata=False),
)

assert params_schema["strict"] is True
assert params_schema["keep_metadata"] is False
assert params_schema["schema"]["source"] == "options"
assert params_schema["schema"]["version"] == schema.version

assert_json_safe(params_default, "params_default")
assert_json_safe(params_schema, "params_schema")

show_ok("Test 6.1 - _options_to_read_parameters")

OK - Test 6.1 - _options_to_read_parameters


### Test 6.2 - `_build_read_trips_summary`

Qué prueba: summary mínimo y estable de OP-07.

In [45]:
summary = _build_read_trips_summary(
    n_rows=3,
    n_columns=12,
    path=Path("/tmp/artifact"),
    storage_format="feather",
    schema_source="metadata",
    schema_mismatch=False,
    dataset_id_status="loaded",
    dataset_id="dset_001",
    artifact_id_status="loaded",
    artifact_id="art_001",
)

assert summary["n_rows"] == 3
assert summary["n_columns"] == 12
assert Path(summary["path"]) == Path("/tmp/artifact")
assert summary["storage_format"] == "feather"
assert summary["schema_source"] == "metadata"
assert summary["schema_mismatch"] is False
assert summary["dataset_id"] == "dset_001"
assert summary["dataset_id_status"] == "loaded"
assert summary["artifact_id"] == "art_001"
assert summary["artifact_id_status"] == "loaded"

assert_json_safe(summary, "read_summary")

show_ok("Test 6.2 - _build_read_trips_summary")

OK - Test 6.2 - _build_read_trips_summary


### Test 6.3 - `_build_issues_summary`, `_build_io_event` y `_append_event` para `read_trips`

Qué prueba: forma mínima de trazabilidad operacional usada por el evento `read_trips`.

In [46]:
issues = [
    Issue(level="info", code="READ.METADATA.VALIDATED_FORCED_FALSE", message="forced false"),
    Issue(level="warning", code="READ.SCHEMA.MISMATCH", message="schema mismatch"),
    Issue(level="warning", code="READ.SCHEMA.MISMATCH", message="schema mismatch again"),
]

issues_summary = _build_issues_summary(issues)

assert issues_summary["counts"]["info"] == 1
assert issues_summary["counts"]["warning"] == 2
assert issues_summary["counts"]["error"] == 0
assert issues_summary["top_codes"][0]["code"] == "READ.SCHEMA.MISMATCH"
assert issues_summary["top_codes"][0]["count"] == 2

parameters = {
    "path": "artifact.golondrina",
    "strict": False,
    "keep_metadata": True,
    "schema": None,
}

summary = {
    "n_rows": 3,
    "n_columns": 12,
    "path": "artifact.golondrina",
    "storage_format": "parquet",
    "schema_source": "metadata",
    "schema_mismatch": False,
    "dataset_id": "dset_001",
    "dataset_id_status": "loaded",
    "artifact_id": "art_001",
    "artifact_id_status": "loaded",
}

event = _build_io_event(
    op="read_trips",
    parameters=parameters,
    summary=summary,
    issues_summary=issues_summary,
)

assert event["op"] == "read_trips"
assert "ts_utc" in event
assert event["parameters"] == parameters
assert event["summary"] == summary
assert event["issues_summary"]["counts"]["warning"] == 2

metadata_in = {"events": [{"op": "previous"}]}
metadata_out = _append_event(metadata_in, event)

assert len(metadata_in["events"]) == 1
assert len(metadata_out["events"]) == 2
assert metadata_out["events"][-1]["op"] == "read_trips"

assert_json_safe(event, "read_event")
assert_json_safe(metadata_out, "metadata_with_read_event")

show_ok("Test 6.3 - issues_summary + read_event + append_event")

OK - Test 6.3 - issues_summary + read_event + append_event
